# Module 5.1: Docker Engine Installation & Non-Root User Provisioning

## Homelab NAS Development - Step 5: Docker Installation

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Install the official upstream Docker Engine repository for ARM64 Linux.
2. Understand containerization architecture (namespaces & cgroups) vs. hypervisor VMs.
3. Expand system swap file to 2048MB to safeguard 4GB RAM against OOM crashes.
4. Configure user permissions to enable non-root Docker interaction.
5. Verify container execution and daemon systemd lifecycle management.

## 5.1.1 Why Containerization for Homelabs?

Traditionally, deploying applications like photo managers, media stream engines, and databases directly onto the host OS led to "dependency hell" - mismatched Python libraries, incompatible PostgreSQL versions, and messy uninstalls.

### Containers vs. Virtual Machines on ARM
- **Virtual Machines**: Emulate virtual hardware, running a complete guest kernel on top of a hypervisor. This introduces severe CPU overhead and consumes hundreds of megabytes of RAM per VM.
- **Docker Containers**: Lightweight processes sharing the **host Linux kernel**. Isolation is enforced using Linux kernel **namespaces** (PID, Mount, Network, IPC) and resource boundaries are managed via **cgroups** (Control Groups). This yields near-zero overhead, ideal for single-board computers like the Raspberry Pi.

### Memory Safeguard: Expanding Swap to 2GB
Raspberry Pi OS Lite 64-bit ships with minimal or no disk swap by default (and newer releases removed the legacy `dphys-swapfile` service). When running multi-container stacks like Immich with heavy PostgreSQL database migrations and machine-learning jobs, provisioning a dedicated **2GB (`2048MB`) swap file** prevents the Linux Out-Of-Memory (OOM) killer from abruptly terminating containers during memory spikes.

## 5.1.2 Step-by-Step Laboratory Guide

### Task 5.1.1: Provision a Persistent 2GB System Swap File
Use standard Linux swap utilities (`fallocate`, `mkswap`, and `swapon`) to allocate a 2GB swap file, lock down its permissions, activate it immediately, and register it in `/etc/fstab` so it persists across reboots:
```bash
# 1. Allocate a contiguous 2GB swap file on the NVMe SSD (protects the MicroSD card from wear!)
sudo fallocate --length 2G /mnt/homelab_storage/swapfile

# 2. Restrict permissions to root-only (required by the Linux kernel for security)
sudo chmod 600 /mnt/homelab_storage/swapfile

# 3. Format the file as a Linux swap area and enable it immediately
sudo mkswap /mnt/homelab_storage/swapfile
sudo swapon /mnt/homelab_storage/swapfile

# 4. Persist the swap file across reboots in /etc/fstab (if not already present)
grep -q '/mnt/homelab_storage/swapfile' /etc/fstab || echo '/mnt/homelab_storage/swapfile none swap sw,nofail 0 0' | sudo tee -a /etc/fstab

# 5. Verify total active swap memory (~2048 MB)
free -m
swapon --show
```

### Task 5.1.2: Install Docker Engine via Official Upstream Repository
Do not use `apt install docker.io`, as it contains outdated Debian package builds. Use Docker's automated installation script:
```bash
curl -fsSL https://get.docker.com -o get-docker.sh
sudo sh get-docker.sh
```

### Task 5.1.3: Manage Docker as a Non-Root User
Allow your admin user to run Docker commands without prepending `sudo`:
```bash
sudo usermod -aG docker $USER
newgrp docker
```

### Task 5.1.4: Enable Docker on System Boot & Run Sanity Test
```bash
sudo systemctl enable docker
sudo systemctl status docker --no-pager
docker run --rm hello-world
```

### 🖼️ Task 5.1.4 Verification: Docker Hello-World Verification

**Verification Objective:** Attach a terminal screenshot showing the complete output of `docker run --rm hello-world` executed without `sudo`.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] All laboratory tasks executed and validated without unhandled errors.
- [ ] Key configurations and credentials confirmed.
- [ ] Verification screenshot captured and referenced.

Proceed to the next module in sequence once this checkpoint is satisfied.